In [21]:
#importing the important libraries
import pandas as pd

#importing dataset 
data= pd.read_csv("netflix_titles[1].csv")
print(data.head())

  show_id     type                  title         director  \
0      s1    Movie   Dick Johnson Is Dead  Kirsten Johnson   
1      s2  TV Show          Blood & Water              NaN   
2      s3  TV Show              Ganglands  Julien Leclercq   
3      s4  TV Show  Jailbirds New Orleans              NaN   
4      s5  TV Show           Kota Factory              NaN   

                                                cast        country  \
0                                                NaN  United States   
1  Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...   South Africa   
2  Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...            NaN   
3                                                NaN            NaN   
4  Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...          India   

           date_added  release_year rating   duration  \
0  September 25, 2021          2020  PG-13     90 min   
1  September 24, 2021          2021  TV-MA  2 Seasons   
2  September 24, 2021        

In [22]:
#shape of the data
print(data.shape,"\n")

#listing all columns
print(data.columns,"\n")

#datatype and non-null values count of each column 
print(data.info(),"\n")

#missing values
print(data.isnull().sum())

(8807, 12) 

Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description'],
      dtype='object') 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB
None 

show_id            0
type               0
title       

In [23]:
#therefore we have missing values in 6 columns

In [24]:
#handling missing values

#director:- too many missing values therefore can't drop or fill with mode
data['director'] = data['director'].fillna('Unkown')

#don't know cast names so filling with unknown
data['cast'] = data['cast'].fillna('Unknown')

#country
data['country'] = data['country'].fillna(data['country'].mode()[0])

#date_added have low missing values so drop can be better
data.dropna(subset=['date_added'], inplace=True)

#rating can be filled with mode
data['rating'] = data['rating'].fillna(data['rating'].mode()[0])

#duration column have 3 missing values and we cannot just simply fill mode value here 
#because we don't know exact duration for any particular show
data.dropna(subset=['duration'], inplace=True)

In [25]:
data.isnull().sum()

show_id         0
type            0
title           0
director        0
cast            0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
description     0
dtype: int64

In [26]:
#converting date_added to date time format
data['date_added'] = pd.to_datetime(data['date_added'], errors='coerce')

data['date_added'].dtype

dtype('<M8[ns]')

In [27]:
#we need to analyse the year trends with monthly patterns therefore,
#extracting year and month differently

data['year_added'] = data['date_added'].dt.year

data['month_added'] = data['date_added'].dt.month

data[['month_added', 'year_added']].head()

,month_added,year_added
0,9.0,2021.0
1,9.0,2021.0
2,9.0,2021.0
3,9.0,2021.0
4,9.0,2021.0


In [28]:
data['year_added'].value_counts()

year_added
2019.0    1999
2020.0    1878
2018.0    1625
2021.0    1498
2017.0    1163
2016.0     416
2015.0      73
2014.0      23
2011.0      13
2013.0      10
2012.0       3
2009.0       2
2008.0       2
2010.0       1
Name: count, dtype: int64

In [29]:
#content addition increased rapid after 2016, 
#peaked around 2019-2020, and slightly declined after that

In [30]:
#cleaning duration column for better analysis
data[['duration_num','duration_unit']] = data['duration'].str.split(' ', expand=True)

#convert numbers into numeric
data['duration_num'] = pd.to_numeric(data['duration_num'])

In [31]:
data.groupby('type')['duration_num'].mean()

type
Movie      99.577187
TV Show     1.751313
Name: duration_num, dtype: float64

In [32]:
#movies average time = 90 min
#shows average no. of seasons = 1.7

In [33]:
year_type = data.groupby(['type','year_added']).size().unstack()

year_type_percentage = year_type.div(year_type.sum(axis=1),axis=0)* 100

year_type_percentage.sort_index()

year_added,2008.0,2009.0,2010.0,2011.0,2012.0,2013.0,2014.0,2015.0,2016.0,2017.0,2018.0,2019.0,2020.0,2021.0
type,,,,,,,,,,,,,,
Movie,0.016319,0.032637,0.016319,0.212141,0.048956,0.097911,0.310052,0.913838,4.095953,13.674935,20.186031,23.237598,20.953003,16.204308
TV Show,0.038790,NaN,NaN,NaN,NaN,0.155159,0.155159,0.659426,6.400310,12.606672,15.050427,22.304112,23.041117,19.588829


In [34]:
#although movie dominates the overall, the proportion of TV shows increased in recent years,
#indicating a shift towards series-based content

In [35]:
print("Total no. of movies and shows:-")
print(data['type'].value_counts(),"\n")

print("Countries having different movis and shows:-")
data['country'].value_counts().head(10)

Total no. of movies and shows:-
type
Movie      6128
TV Show    2666
Name: count, dtype: int64 

Countries having different movis and shows:-


country
United States     3639
India              972
United Kingdom     418
Japan              244
South Korea        199
Canada             181
Spain              145
France             124
Mexico             110
Egypt              106
Name: count, dtype: int64

In [36]:
#so netflix have significantly movies than TV Shows, indicating movie-heavy catalog
#The united states contributes the highest content, followed by India and the UK, 
#showing netflix still US-dominated but expanding globally

In [37]:
data['rating'].value_counts()

rating
TV-MA       3209
TV-14       2157
TV-PG        861
R            799
PG-13        490
TV-Y7        333
TV-Y         306
PG           287
TV-G         220
NR            79
G             41
TV-Y7-FV       6
NC-17          3
UR             3
Name: count, dtype: int64

In [38]:
#TV-MA is the most common rating, indicating Netflix primarily targets mature audience.

In [39]:
data['listed_in'].value_counts().head(10)

listed_in
Dramas, International Movies                        362
Documentaries                                       359
Stand-Up Comedy                                     334
Comedies, Dramas, International Movies              274
Dramas, Independent Movies, International Movies    252
Kids' TV                                            219
Children & Family Movies                            215
Children & Family Movies, Comedies                  201
Documentaries, International Movies                 186
Dramas, International Movies, Romantic Movies       180
Name: count, dtype: int64

In [40]:
#Dramas and international content dominates Netflix, 
#indicating  a focus on diverse storytelling